In [2]:
import pandas as pd
import numpy as np

movies = pd.read_csv("./data/movies.csv")
tags = pd.read_csv("./data/tags.csv")

In [3]:
tags = tags[["movieId", "tag"]].copy()
tags.head()

,movieId,tag
0,260,good vs evil
1,260,Harrison Ford
2,260,sci-fi
3,1221,Al Pacino
4,1221,mafia


In [4]:
print(movies.shape)
print(tags.shape)

movies.head()

(86537, 3)
(2328315, 2)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
tags = tags.dropna(subset=["tag"]).copy()
tags["tag"] = tags["tag"].str.lower().str.strip()
tags = tags[tags["tag"] != ""]
tags = tags.drop_duplicates()

print(tags.shape)
print(tags["movieId"].nunique())
tags["tag"].value_counts().head(20)

(1142592, 2)
53452


tag
bd-r                3948
murder              3810
woman director      3602
independent film    2861
violence            2328
comedy              2306
drama               2106
death               1968
romance             1868
friendship          1856
revenge             1826
based on a book     1719
love                1718
funny               1633
nudity (topless)    1630
police              1526
boring              1525
criterion           1468
blood               1434
sex                 1415
Name: count, dtype: int64

In [6]:
tags_grouped = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

tags_grouped.head()

,movieId,tag
0,1,animation friendship toys disney pixar cgi cla...
1,2,animals based on a book fantasy magic board ga...
2,3,sequel moldy old old age old men wedding old p...
3,4,characters chick flick girl movie revenge clv ...
4,5,family pregnancy wedding 4th wall aging baby d...


In [7]:
movies_with_tags = movies.merge(tags_grouped, on="movieId", how="left")
movies_with_tags.head()
movies_with_tags["tag"].isna().sum()
movies_with_tags.info()

<class 'pandas.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  86537 non-null  int64
 1   title    86537 non-null  str  
 2   genres   86537 non-null  str  
 3   tag      53452 non-null  str  
dtypes: int64(1), str(3)
memory usage: 2.6 MB


In [8]:
movies_with_tags["genres_text"] = (
    movies_with_tags["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
)

movies_with_tags["content"] = (
    (movies_with_tags["genres_text"].fillna("") + " ") * 3 +
    movies_with_tags["tag"].fillna("")
).str.strip()

movies_with_tags["title_clean"] = (
    movies_with_tags["title"]
    .str.replace(r"\s*\(\d{4}\)$", "", regex=True)
    .str.lower()
    .str.strip()
)

movies_with_tags["content_word_count"] = (
    movies_with_tags["content"]
    .str.split()
    .str.len()
)

movies_with_tags.head()

,movieId,title,genres,tag,genres_text,content,title_clean,content_word_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,animation friendship toys disney pixar cgi cla...,adventure animation children comedy fantasy,adventure animation children comedy fantasy ad...,toy story,838
1,2,Jumanji (1995),Adventure|Children|Fantasy,animals based on a book fantasy magic board ga...,adventure children fantasy,adventure children fantasy adventure children ...,jumanji,726
2,3,Grumpier Old Men (1995),Comedy|Romance,sequel moldy old old age old men wedding old p...,comedy romance,comedy romance comedy romance comedy romance s...,grumpier old men,62
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,characters chick flick girl movie revenge clv ...,comedy drama romance,comedy drama romance comedy drama romance come...,waiting to exhale,27
4,5,Father of the Bride Part II (1995),Comedy,family pregnancy wedding 4th wall aging baby d...,comedy,comedy comedy comedy family pregnancy wedding ...,father of the bride part ii,67


In [9]:
ratings = pd.read_csv("./data/ratings.csv", usecols=["movieId", "rating"])
print(ratings.shape)

ratings_summary = (
    ratings.groupby("movieId")
    .agg(
        mean_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

ratings_summary.head()

(33832162, 2)


,movieId,mean_rating,rating_count
0,1,3.893508,76813
1,2,3.278179,30209
2,3,3.171271,15820
3,4,2.868395,3028
4,5,3.076957,15801


In [10]:
movies_model = movies_with_tags[
    ~(
        (movies_with_tags["genres"] == "(no genres listed)") &
        (movies_with_tags["tag"].isna())
    )
].copy()

movies_model = movies_model[movies_model["content_word_count"] >= 3].copy()
movies_model = movies_model.reset_index(drop=True)

movies_model = movies_model.merge(ratings_summary, on="movieId", how="left")
movies_model["mean_rating"] = movies_model["mean_rating"].fillna(0)
movies_model["rating_count"] = movies_model["rating_count"].fillna(0)
movies_model.head()

,movieId,title,genres,tag,genres_text,content,title_clean,content_word_count,mean_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,animation friendship toys disney pixar cgi cla...,adventure animation children comedy fantasy,adventure animation children comedy fantasy ad...,toy story,838,3.893508,76813.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,animals based on a book fantasy magic board ga...,adventure children fantasy,adventure children fantasy adventure children ...,jumanji,726,3.278179,30209.0
2,3,Grumpier Old Men (1995),Comedy|Romance,sequel moldy old old age old men wedding old p...,comedy romance,comedy romance comedy romance comedy romance s...,grumpier old men,62,3.171271,15820.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,characters chick flick girl movie revenge clv ...,comedy drama romance,comedy drama romance comedy drama romance come...,waiting to exhale,27,2.868395,3028.0
4,5,Father of the Bride Part II (1995),Comedy,family pregnancy wedding 4th wall aging baby d...,comedy,comedy comedy comedy family pregnancy wedding ...,father of the bride part ii,67,3.076957,15801.0


In [11]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    min_df=2,
    ngram_range=(1, 2)
)

X_tfidf = tfidf.fit_transform(movies_model["content"])
print(X_tfidf.shape)

nn_model = NearestNeighbors(metric="cosine", algorithm="brute")
nn_model.fit(X_tfidf)

(82267, 5000)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [12]:
def recommend_movies(title, movies_df, feature_matrix, model, n=5, candidate_pool=20, min_ratings=50):
    query = title.lower().strip()

    matches = movies_df[movies_df["title"].str.lower() == query]

    if matches.empty:
        matches = movies_df[movies_df["title_clean"] == query]

    if matches.empty:
        matches = movies_df[movies_df["title_clean"].str.contains(query, na=False)]

    if matches.empty:
        return f"Movie '{title}' not found."

    if len(matches) > 1:
        return matches[["title", "genres"]].head(10)

    idx = matches.index[0]

    distances, indices = model.kneighbors(
        feature_matrix[idx],
        n_neighbors=candidate_pool + 1
    )

    rec_indices = indices.flatten()[1:]
    rec_distances = distances.flatten()[1:]

    recommendations = movies_df.iloc[rec_indices][
        ["title", "genres", "tag", "content_word_count", "mean_rating", "rating_count"]
    ].copy()

    recommendations["distance"] = rec_distances

    recommendations = recommendations[
        ~(
            (recommendations["genres"] == "(no genres listed)") &
            (recommendations["tag"].isna())
        )
    ]

    recommendations = recommendations[recommendations["content_word_count"] >= 3]
    recommendations = recommendations[recommendations["rating_count"] >= min_ratings]

    recommendations["weighted_score"] = (
        recommendations["mean_rating"] * np.log1p(recommendations["rating_count"])
    )

    recommendations = recommendations.sort_values(
        by=["weighted_score", "distance"],
        ascending=[False, True]
    )

    return recommendations[
        ["title", "genres", "mean_rating", "rating_count", "weighted_score", "distance"]
    ].head(n)

In [17]:
recommend_movies("Bridget Jones's Diary (2001)", movies_model, X_tfidf, nn_model)

,title,genres,mean_rating,rating_count,weighted_score,distance
16,Sense and Sensibility (1995),Drama|Romance,3.945897,25248.0,39.997747,0.695370
1273,When Harry Met Sally... (1989),Comedy|Romance,3.835569,28705.0,39.371581,0.696848
534,Sleepless in Seattle (1993),Comedy|Drama|Romance,3.514057,33436.0,36.607399,0.690016
6818,Love Actually (2003),Comedy|Drama|Romance,3.701663,18878.0,36.445857,0.653728
5270,About a Boy (2002),Comedy|Drama|Romance,3.628116,13359.0,34.467175,0.666605


In [16]:
results = recommend_movies("Clueless", movies_model, X_tfidf, nn_model)

results.merge(
    movies_model[["title", "content"]],
    on="title",
    how="left"
)

,title,genres,mean_rating,rating_count,weighted_score,distance,content
0,10 Things I Hate About You (1999),Comedy|Romance,3.560852,18019.0,34.893639,0.568138,comedy romance comedy romance comedy romance r...
1,She's All That (1999),Comedy|Romance,3.012979,7743.0,26.980248,0.574623,comedy romance comedy romance comedy romance t...
2,Youth in Revolt (2009),Comedy|Drama|Romance,3.364026,1401.0,24.374570,0.625152,comedy drama romance comedy drama romance come...
3,The DUFF (2015),Comedy,3.331284,1301.0,23.890823,0.627489,comedy comedy comedy romantic comedy funny hig...
4,Pride and Prejudice (1940),Comedy|Drama|Romance,3.659161,644.0,23.672032,0.622939,comedy drama romance comedy drama romance come...
